# Q2. Classification Model with Spark MLlib
Use Case: Predicting customer churn in a telecom dataset.

## Steps
1. Load telecom dataset into Spark.
2. Preprocess columns and build feature vector.
3. Split data into train/test.
4. Train Logistic Regression classifier.
5. Evaluate Accuracy, Precision, Recall, and F1.

In [33]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import expr

In [34]:
spark = SparkSession.builder.appName("ChurnPrediction").getOrCreate()
spark

26/04/26 19:03:15 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [35]:
# Update the path if needed
df = spark.read.csv("telecom.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5, truncate=False)

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)

+----------+------+-------------+-------+----------+------+------------+---------

In [36]:
# Convert TotalCharges safely to numeric and fill invalid values
df = df.withColumn("TotalCharges", expr("try_cast(TotalCharges AS DOUBLE)"))
df = df.na.fill(0, subset=["TotalCharges"])

# Encode label and categorical features
label_indexer = StringIndexer(inputCol="Churn", outputCol="label", handleInvalid="keep")
gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_index", handleInvalid="keep")
contract_indexer = StringIndexer(inputCol="Contract", outputCol="contract_index", handleInvalid="keep")

df = label_indexer.fit(df).transform(df)
df = gender_indexer.fit(df).transform(df)
df = contract_indexer.fit(df).transform(df)

In [37]:
assembler = VectorAssembler(
    inputCols=["tenure", "MonthlyCharges", "TotalCharges", "gender_index", "contract_index"],
    outputCol="features",
    handleInvalid="skip"
)
data = assembler.transform(df).select("features", "label")
data.show(5, truncate=False)

+---------------------------+-----+
|features                   |label|
+---------------------------+-----+
|[1.0,29.85,29.85,1.0,0.0]  |0.0  |
|[34.0,56.95,1889.5,0.0,2.0]|0.0  |
|[2.0,53.85,108.15,0.0,0.0] |1.0  |
|[45.0,42.3,1840.75,0.0,2.0]|0.0  |
|[2.0,70.7,151.65,1.0,0.0]  |1.0  |
+---------------------------+-----+
only showing top 5 rows


In [38]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)
print("Train count:", train_data.count())
print("Test count:", test_data.count())

Train count: 5698
Test count: 1345


In [39]:
lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train_data)
predictions = model.transform(test_data)
predictions.select("label", "prediction", "probability").show(10, truncate=False)

+-----+----------+---------------------------------------------------------------+
|label|prediction|probability                                                    |
+-----+----------+---------------------------------------------------------------+
|0.0  |0.0       |[0.8222106338344654,0.17778936615323784,1.2296917866181026E-11]|
|0.0  |0.0       |[0.650464629261622,0.34953537073557417,2.8038518607557894E-12] |
|0.0  |0.0       |[0.5926237151689202,0.4073762848294466,1.6333248507052362E-12] |
|0.0  |0.0       |[0.7314035341083349,0.2685964658849203,6.744722265391298E-12]  |
|1.0  |0.0       |[0.7300213273848694,0.26997867260846303,6.667693771826275E-12] |
|0.0  |0.0       |[0.7324619367962008,0.2675380631977079,6.0913942808170245E-12] |
|0.0  |0.0       |[0.7319109704976097,0.26808902949632685,6.063496614603368E-12] |
|0.0  |0.0       |[0.7283567749248879,0.2716432250685357,6.5763354699553704E-12] |
|1.0  |0.0       |[0.7278004962331188,0.27219950376033514,6.546141290113938E-12] |
|0.0

In [40]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
accuracy = evaluator.setMetricName("accuracy").evaluate(predictions)
precision = evaluator.setMetricName("weightedPrecision").evaluate(predictions)
recall = evaluator.setMetricName("weightedRecall").evaluate(predictions)
f1 = evaluator.setMetricName("f1").evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.7911
Precision: 0.7763
Recall: 0.7911
F1 Score: 0.7784


In [41]:
model.write().overwrite().save("churn_model")
print("Model saved to: churn_model")

Model saved to: churn_model


## Q2 Report Summary (Classification)
### Aim
To build a Spark MLlib classification model for predicting customer churn from telecom customer attributes.
### Method Summary
The dataset was loaded into Spark, required fields were cleaned, categorical columns were indexed, and numerical plus indexed features were combined using VectorAssembler. The data was split into train and test subsets, and Logistic Regression was trained using the training partition.
### Result Summary
The model produced consistent performance on the test split with Accuracy = 0.7911, Precision = 0.7763, Recall = 0.7911, and F1 Score = 0.7784.
### Inference
The model shows reliable baseline churn prediction performance and captures useful patterns from contract type, charges, and tenure-based behavior. Since evaluation metrics are close to each other, the model quality is balanced and suitable for initial decision support.
### Conclusion
The objective of churn classification was achieved successfully using Spark MLlib. The trained model was saved and can be used for further deployment or tuning.

---\n
# Q3. Regression Model with Spark MLlib\n
Use Case: Predicting house prices using a real estate dataset.

# Q3. Regression Model with Spark MLlib
Use Case: Predicting house prices using a real estate dataset.

## Steps
1. Load housing data in Spark.
2. Clean and cast numeric columns.
3. Assemble and scale features.
4. Train Linear Regression model.
5. Evaluate using RMSE and R2.

In [42]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [43]:
spark = SparkSession.builder.appName("HousingRegression").getOrCreate()
spark

26/04/26 19:03:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [44]:
# Update file path if needed
df = spark.read.csv("re.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5, truncate=False)

root
 |-- Name: string (nullable = true)
 |-- Property Title: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- Total_Area: double (nullable = true)
 |-- Price_per_SQFT: double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Baths: double (nullable = true)
 |-- Balcony: string (nullable = true)

+----------+--------------+---------+------------------------+----------+-----------------+--------------------+-----+-------+
|Name      |Property Title|Price    |Location                |Total_Area|Price_per_SQFT   |Description         |Baths|Balcony|
+----------+--------------+---------+------------------------+----------+-----------------+--------------------+-----+-------+
|Property_1|2 BHK         |3907000.0|Electronic City Phase II|1056.0    |3699.810606060606|Super built-up  Area|2.0  |Yes    |
|Property_2|4 Bedroom     |1.2E7    |Chikka Tirupathi        |2600.0    |4615.384615384615|Plot  Area          |5.0  |Ye

In [45]:
# Robust numeric casting for Spark-inferred columns
df = df.withColumn("Price", col("Price").cast("double"))
df = df.withColumn("Total_Area", col("Total_Area").cast("double"))
df = df.withColumn("Price_per_SQFT", col("Price_per_SQFT").cast("double"))
df = df.withColumn("Baths", col("Baths").cast("double"))
df = df.withColumn(
    "Balcony",
    when(col("Balcony") == "Yes", 1.0)
    .when(col("Balcony") == "No", 0.0)
    .otherwise(None)
    .cast("double")
)

df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Property Title: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- Total_Area: double (nullable = true)
 |-- Price_per_SQFT: double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Baths: double (nullable = true)
 |-- Balcony: double (nullable = true)



In [46]:
assembler = VectorAssembler(
    inputCols=["Total_Area", "Price_per_SQFT", "Baths", "Balcony"],
    outputCol="features",
    handleInvalid="skip"
)
data = assembler.transform(df).dropna(subset=["features", "Price"])

In [47]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
scaler_model = scaler.fit(data)
data = scaler_model.transform(data)

In [48]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)
print("Train count:", train_data.count())
print("Test count:", test_data.count())

Train count: 10626
Test count: 2586


In [49]:
if test_data.count() == 0:
    print("Test data is empty. Cannot evaluate the model.")
else:
    lr = LinearRegression(featuresCol="scaled_features", labelCol="Price")
    model = lr.fit(train_data)
    predictions = model.transform(test_data)
    predictions.select("Price", "prediction").show(10, truncate=False)

    rmse = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse").evaluate(predictions)
    r2 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="r2").evaluate(predictions)

    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")

26/04/26 19:03:19 WARN Instrumentation: [ae27f65c] regParam is zero, which might cause numerical instability and overfitting.


+------------------+--------------------+
|Price             |prediction          |
+------------------+--------------------+
|9000000.0         |1.3260845152406074E7|
|1.6E7             |1.8752354410303976E7|
|6500000.0         |6838876.158893209   |
|1.75E7            |5.3208545067586906E7|
|1.16E7            |1.1559517055177558E7|
|1.11E7            |1.192813083415144E7 |
|9500000.0         |1.0988980052966982E7|
|4113000.0000000005|3828686.1634395868  |
|2.99E7            |2.2651799935333394E7|
|5600000.0         |7185929.887893265   |
+------------------+--------------------+
only showing top 10 rows
RMSE: 10487394.2297
R2 Score: 0.5369


In [50]:
if "model" in locals():
    model.write().overwrite().save("house_price_model")
    print("Model saved to: house_price_model")
else:
    print("Model not trained due to empty test split.")

Model saved to: house_price_model


## Q3 Report Summary (Regression)
### Aim
To build a Spark MLlib regression model for predicting house prices from structured real-estate features.
### Method Summary
The housing dataset was loaded and numerically cast, balcony information was encoded, features were assembled, and scaling was applied using StandardScaler. Linear Regression was trained on the training split and evaluated on the test split.
### Result Summary
The regression model achieved RMSE = 10487394.2297 and R2 Score = 0.5369 on the test partition.
### Inference
The model captures the general direction of price variation but still has notable error for high-variance and extreme-price properties. The positive R2 indicates meaningful explanatory power, while RMSE suggests further feature refinement is needed for stronger precision.
### Conclusion
The house-price regression pipeline was completed successfully in Spark MLlib, and the trained model was saved for future improvement and comparative experiments.